In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import joblib
import pickle

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [6]:
customer_df = pd.read_csv("data/olist_customers_dataset.csv")
orders_df = pd.read_csv("data/olist_orders_dataset.csv")
items_df = pd.read_csv("data/olist_order_items_dataset.csv")
products_df = pd.read_csv("data/olist_products_dataset.csv")
transl_df=pd.read_csv("data/product_category_name_translation.csv")

### cleaning

### duplicate ,datetime, null

### customers

In [7]:
print(customer_df.info())
print(customer_df.isnull().sum())
print(customer_df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB
None
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
0


### orders

In [8]:
print(orders_df.info())
print(orders_df.isnull().sum())
print(orders_df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB
None
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date

In [9]:
orders_df['order_purchase_timestamp'] = pd.to_datetime(orders_df['order_purchase_timestamp'])
orders_df['order_approved_at'] = pd.to_datetime(orders_df['order_approved_at'])
orders_df['order_delivered_carrier_date'] = pd.to_datetime(orders_df['order_delivered_carrier_date'])
orders_df['order_delivered_customer_date'] = pd.to_datetime(orders_df['order_delivered_customer_date'])
orders_df['order_estimated_delivery_date'] = pd.to_datetime(orders_df['order_estimated_delivery_date'])

In [10]:
orders_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


In [11]:
orders_df = orders_df.dropna()
print(orders_df.isnull().sum())

order_id                         0
customer_id                      0
order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
dtype: int64


### items

In [12]:
print(items_df.info())
print(items_df.isnull().sum())
print(items_df.duplicated().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB
None
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
0


In [13]:
items_df['shipping_limit_date'] = pd.to_datetime(items_df['shipping_limit_date'])
items_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  object        
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  object        
 3   seller_id            112650 non-null  object        
 4   shipping_limit_date  112650 non-null  datetime64[ns]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(3)
memory usage: 6.0+ MB


### products_df

In [14]:
print(products_df.info())
print(products_df.isnull().sum())
print(products_df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB
None
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
prod

In [15]:
products_df.dropna(inplace=True)

### translation

In [16]:
print(transl_df.info())
print(transl_df.isnull().sum())
print(transl_df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     object
 1   product_category_name_english  71 non-null     object
dtypes: object(2)
memory usage: 1.2+ KB
None
product_category_name            0
product_category_name_english    0
dtype: int64
0


### merging dataframes

In [17]:
df=orders_df.merge(customer_df,on="customer_id",how="left")
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP


In [18]:
df=df.merge(items_df,on="order_id",how="left")
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72


In [19]:
df=df.merge(products_df,on="product_id",how="left")
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,45.00,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,19.90,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0


In [20]:
df=df.merge(transl_df,on="product_category_name",how="left")
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery


In [21]:
df.describe()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_zip_code_prefix,order_item_id,shipping_limit_date,price,freight_value,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,110180,110180,110180,110180,110180,110180.000000,110180.000000,110180,110180.000000,110180.000000,108643.000000,108643.000000,108643.000000,108643.000000,108643.000000,108643.000000,108643.000000
mean,2018-01-02 03:09:05.618805760,2018-01-02 13:40:14.266463744,2018-01-05 09:57:40.877328128,2018-01-14 14:29:00.951887872,2018-01-25 22:27:00.620802304,35155.850826,1.198212,2018-01-08 17:38:11.457660160,119.976817,19.948713,48.807691,787.319781,2.210810,2095.564178,30.197629,16.588312,23.030467
min,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-10-04 00:00:00,1003.000000,1.000000,2016-09-19 23:11:33,0.850000,0.000000,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000
25%,2017-09-15 08:33:15.249999872,2017-09-15 14:25:19.750000128,2017-09-18 22:46:46.249999872,2017-09-26 20:26:16.249999872,2017-10-06 00:00:00,11310.000000,1.000000,2017-09-21 15:05:55,39.900000,13.080000,42.000000,348.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,2018-01-21 10:16:58.500000,2018-01-22 13:56:54,2018-01-24 18:59:21,2018-02-02 21:14:55,2018-02-16 00:00:00,24344.000000,1.000000,2018-01-26 20:27:10.500000,74.900000,16.260000,52.000000,603.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,2018-05-05 15:59:52,2018-05-05 22:30:47.500000,2018-05-08 14:20:00,2018-05-15 20:19:18,2018-05-28 00:00:00,59066.000000,1.000000,2018-05-10 20:19:29.249999872,134.170000,21.150000,57.000000,987.000000,3.000000,1800.000000,38.000000,20.000000,30.000000
max,2018-08-29 15:00:37,2018-08-29 15:10:26,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-10-25 00:00:00,99980.000000,21.000000,2020-04-09 22:35:08,6735.000000,409.680000,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000
std,NaN,NaN,NaN,NaN,NaN,29901.214959,0.706726,NaN,182.309380,15.699220,10.008580,651.346697,1.721835,3744.019308,16.157227,13.433018,11.696960


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110180 entries, 0 to 110179
Data columns (total 27 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       110180 non-null  object        
 1   customer_id                    110180 non-null  object        
 2   order_status                   110180 non-null  object        
 3   order_purchase_timestamp       110180 non-null  datetime64[ns]
 4   order_approved_at              110180 non-null  datetime64[ns]
 5   order_delivered_carrier_date   110180 non-null  datetime64[ns]
 6   order_delivered_customer_date  110180 non-null  datetime64[ns]
 7   order_estimated_delivery_date  110180 non-null  datetime64[ns]
 8   customer_unique_id             110180 non-null  object        
 9   customer_zip_code_prefix       110180 non-null  int64         
 10  customer_city                  110180 non-null  object        
 11  

In [23]:
df.drop(df[df['order_status']=='canceled'].index,inplace=True)

In [24]:
df.drop("order_status", axis=1, inplace=True)

In [25]:
df.head()

,order_id,customer_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,...,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,...,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,...,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,...,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,...,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,...,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery


In [26]:
df.shape

(110173, 26)

In [27]:
df.isnull().sum()

order_id                            0
customer_id                         0
order_purchase_timestamp            0
order_approved_at                   0
order_delivered_carrier_date        0
order_delivered_customer_date       0
order_estimated_delivery_date       0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
order_item_id                       0
product_id                          0
seller_id                           0
shipping_limit_date                 0
price                               0
freight_value                       0
product_category_name            1537
product_name_lenght              1537
product_description_lenght       1537
product_photos_qty               1537
product_weight_g                 1537
product_length_cm                1537
product_height_cm                1537
product_width_cm                 1537
product_category_name_english    1559
dtype: int64

In [28]:
df.dropna(inplace=True)

In [29]:
nec_cols = [
    'order_id',
    'customer_id',
    'order_purchase_timestamp',
'customer_unique_id',
'customer_state',
'order_item_id',
'product_id',
'seller_id',
'price',
'freight_value',
'product_weight_g',
'product_length_cm',
'product_height_cm',
'product_width_cm',
'product_category_name_english']

df.drop(columns=['order_approved_at','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date','customer_zip_code_prefix','customer_city','customer_state','order_item_id','product_id','seller_id','shipping_limit_date'],inplace=True)

In [30]:
df.head()

,order_id,customer_id,order_purchase_timestamp,customer_unique_id,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02 10:56:33,7c396fd4830fd04220f754e42b4e5bff,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24 20:41:37,af07308b275d755c9edb36a90c618231,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 08:38:49,3a653a41f6f9fc3d2a113cf8398680e8,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18 19:28:06,7c142cf63193a1473d2e66489a9ae977,45.00,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13 21:18:39,72632f0f9dd73dfee390c9b22eb56dd6,19.90,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery


In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 108614 entries, 0 to 110179
Data columns (total 15 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       108614 non-null  object        
 1   customer_id                    108614 non-null  object        
 2   order_purchase_timestamp       108614 non-null  datetime64[ns]
 3   customer_unique_id             108614 non-null  object        
 4   price                          108614 non-null  float64       
 5   freight_value                  108614 non-null  float64       
 6   product_category_name          108614 non-null  object        
 7   product_name_lenght            108614 non-null  float64       
 8   product_description_lenght     108614 non-null  float64       
 9   product_photos_qty             108614 non-null  float64       
 10  product_weight_g               108614 non-null  float64       
 11  produ

In [32]:
df = df.sort_values(by=["customer_unique_id", "order_purchase_timestamp"])

In [33]:
df.head()

,order_id,customer_id,order_purchase_timestamp,customer_unique_id,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
58438,e22acc9c116caa3f2b7121bbb380d08e,fadbb3709178fc513abc1b2670aa1ad2,2018-05-10 10:56:27,0000366f3b9a7992bf8c76cfdf3221e2,129.90,12.00,cama_mesa_banho,60.0,236.0,1.0,1500.0,34.0,7.0,32.0,bed_bath_table
81788,3594e05a005ac4d06a72673270ef9ec9,4cb282e167ae9234755102258dd52ee8,2018-05-07 11:11:27,0000b849f77a49e4a4ce2b2a4ca5be3f,18.90,8.29,beleza_saude,56.0,635.0,1.0,375.0,26.0,11.0,18.0,health_beauty
29303,b33ec3b699337181488304f362a6b734,9b3932a6253894a02c1df9d19004239f,2017-03-10 21:05:03,0000f46a3911fa3c0805444483337064,69.00,17.22,papelaria,49.0,177.0,3.0,1500.0,25.0,50.0,35.0,stationery
109100,41272756ecddd9a9ed0180413cc22fb6,914991f0c02ef0843c0e7010c819d642,2017-10-12 20:29:41,0000f6ccb0745a6a4b88665a16c9f078,25.99,17.63,telefonia,43.0,1741.0,5.0,150.0,19.0,5.0,11.0,telephony
46020,d957021f1127559cd947b62533f484f7,47227568b10f5f58a524a75507e6992c,2017-11-14 19:45:42,0004aac84e0df4da2b147fca70cf8255,180.00,16.89,telefonia,58.0,794.0,3.0,6050.0,16.0,3.0,11.0,telephony


In [34]:
df["next_category"] = df.groupby("customer_unique_id")[
    "product_category_name_english"
].shift(-1)
df = df.dropna(subset=["next_category"])
df.head(20)

,order_id,customer_id,order_purchase_timestamp,customer_unique_id,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,next_category
7571,44e608f2db00c74a1fe329de44416a4e,a81ebb9b32f102298c0c89635b4b3154,2018-02-28 11:15:41,00053a61a98854899e70ed204dd4bafe,191.00,18.59,esporte_lazer,42.0,190.0,1.0,1092.0,30.0,26.0,22.0,sports_leisure,sports_leisure
84861,c6d61340bd8baeedca7cc8e7f7ec07e9,455f2e2988eaf87d7e2ba33b0a57969f,2017-08-17 19:10:33,000de6019bb59f34c099a907c151d855,89.90,10.31,cama_mesa_banho,58.0,188.0,1.0,1200.0,44.0,2.0,35.0,bed_bath_table,bed_bath_table
79312,87440e08790d85796f5b8bc9f5ed2707,4b95f958af9c866353ae1108d8ebd023,2018-07-26 09:43:52,000fbf0473c10fc1ab6f8d2d286ce20c,119.87,19.14,instrumentos_musicais,37.0,182.0,1.0,250.0,24.0,10.0,11.0,musical_instruments,musical_instruments
79313,87440e08790d85796f5b8bc9f5ed2707,4b95f958af9c866353ae1108d8ebd023,2018-07-26 09:43:52,000fbf0473c10fc1ab6f8d2d286ce20c,119.87,19.14,instrumentos_musicais,37.0,182.0,1.0,250.0,24.0,10.0,11.0,musical_instruments,toys
79314,87440e08790d85796f5b8bc9f5ed2707,4b95f958af9c866353ae1108d8ebd023,2018-07-26 09:43:52,000fbf0473c10fc1ab6f8d2d286ce20c,23.03,19.14,brinquedos,37.0,512.0,1.0,400.0,20.0,11.0,16.0,toys,toys
72346,533dbcda0a703be171113573af8b3467,5247e5c7e9037e74343f13bbd8800a6a,2017-08-31 17:12:56,001147e649a7b1afd577e873841632dd,85.00,21.08,utilidades_domesticas,54.0,851.0,1.0,3100.0,45.0,28.0,26.0,housewares,housewares
26366,c97e64988f3d2b091838bb2815fc246b,5e51f41c2c9cc612e4d63c9fd18b28c6,2018-08-02 18:23:51,0015752e079902b12cd00b9b7596276b,29.90,7.51,malas_acessorios,57.0,605.0,9.0,210.0,17.0,22.0,11.0,luggage_accessories,luggage_accessories
37656,6fc8cd5f4cfbea0b694434a10475fd6f,b56f0c76d060167be49b4f8208576cca,2017-08-17 12:55:47,001926cef41060fae572e2e7b30bd2a4,18.90,9.43,eletronicos,52.0,390.0,1.0,150.0,20.0,13.0,18.0,electronics,computers_accessories
35677,aa14b8f4567fef1be1a8912ca010f1c7,cec800e76b1cc898de17926aa9e1e146,2018-08-24 21:17:00,001928b561575b2821c92254a2327d06,109.90,25.38,moveis_sala,51.0,598.0,2.0,1350.0,44.0,14.0,34.0,furniture_living_room,bed_bath_table
25960,dc9b66f791e6bc0d3a44e2c513a1d117,cd030cc7d9fe30c0ca7889aeffa326a4,2018-07-24 11:09:11,0025795df7a7d077c4c90162fa820085,79.00,34.79,papelaria,50.0,201.0,2.0,8550.0,60.0,20.0,29.0,stationery,stationery


In [35]:
df.drop_duplicates()
df.shape

(16557, 16)

In [36]:
df.head()

,order_id,customer_id,order_purchase_timestamp,customer_unique_id,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,next_category
7571,44e608f2db00c74a1fe329de44416a4e,a81ebb9b32f102298c0c89635b4b3154,2018-02-28 11:15:41,00053a61a98854899e70ed204dd4bafe,191.00,18.59,esporte_lazer,42.0,190.0,1.0,1092.0,30.0,26.0,22.0,sports_leisure,sports_leisure
84861,c6d61340bd8baeedca7cc8e7f7ec07e9,455f2e2988eaf87d7e2ba33b0a57969f,2017-08-17 19:10:33,000de6019bb59f34c099a907c151d855,89.90,10.31,cama_mesa_banho,58.0,188.0,1.0,1200.0,44.0,2.0,35.0,bed_bath_table,bed_bath_table
79312,87440e08790d85796f5b8bc9f5ed2707,4b95f958af9c866353ae1108d8ebd023,2018-07-26 09:43:52,000fbf0473c10fc1ab6f8d2d286ce20c,119.87,19.14,instrumentos_musicais,37.0,182.0,1.0,250.0,24.0,10.0,11.0,musical_instruments,musical_instruments
79313,87440e08790d85796f5b8bc9f5ed2707,4b95f958af9c866353ae1108d8ebd023,2018-07-26 09:43:52,000fbf0473c10fc1ab6f8d2d286ce20c,119.87,19.14,instrumentos_musicais,37.0,182.0,1.0,250.0,24.0,10.0,11.0,musical_instruments,toys
79314,87440e08790d85796f5b8bc9f5ed2707,4b95f958af9c866353ae1108d8ebd023,2018-07-26 09:43:52,000fbf0473c10fc1ab6f8d2d286ce20c,23.03,19.14,brinquedos,37.0,512.0,1.0,400.0,20.0,11.0,16.0,toys,toys


In [37]:
df['product_category_name_english'].value_counts()

product_category_name_english
furniture_decor                          2262
bed_bath_table                           2204
computers_accessories                    1329
housewares                               1269
sports_leisure                           1240
                                         ... 
furniture_mattress_and_upholstery           3
cds_dvds_musicals                           2
fashion_childrens_clothes                   2
music                                       1
small_appliances_home_oven_and_coffee       1
Name: count, Length: 70, dtype: int64

In [38]:

le_category = LabelEncoder()
df["product_category_name_english"] = le_category.fit_transform(df["product_category_name_english"])

le_next = LabelEncoder()
df["next_category"] = le_next.fit_transform(df["next_category"])

joblib.dump(le_category, "le_category.pkl")
joblib.dump(le_next, "le_next.pkl")


['le_next.pkl']

In [39]:
df.head()

,order_id,customer_id,order_purchase_timestamp,customer_unique_id,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,next_category
7571,44e608f2db00c74a1fe329de44416a4e,a81ebb9b32f102298c0c89635b4b3154,2018-02-28 11:15:41,00053a61a98854899e70ed204dd4bafe,191.00,18.59,esporte_lazer,42.0,190.0,1.0,1092.0,30.0,26.0,22.0,64,63
84861,c6d61340bd8baeedca7cc8e7f7ec07e9,455f2e2988eaf87d7e2ba33b0a57969f,2017-08-17 19:10:33,000de6019bb59f34c099a907c151d855,89.90,10.31,cama_mesa_banho,58.0,188.0,1.0,1200.0,44.0,2.0,35.0,7,7
79312,87440e08790d85796f5b8bc9f5ed2707,4b95f958af9c866353ae1108d8ebd023,2018-07-26 09:43:52,000fbf0473c10fc1ab6f8d2d286ce20c,119.87,19.14,instrumentos_musicais,37.0,182.0,1.0,250.0,24.0,10.0,11.0,56,55
79313,87440e08790d85796f5b8bc9f5ed2707,4b95f958af9c866353ae1108d8ebd023,2018-07-26 09:43:52,000fbf0473c10fc1ab6f8d2d286ce20c,119.87,19.14,instrumentos_musicais,37.0,182.0,1.0,250.0,24.0,10.0,11.0,56,67
79314,87440e08790d85796f5b8bc9f5ed2707,4b95f958af9c866353ae1108d8ebd023,2018-07-26 09:43:52,000fbf0473c10fc1ab6f8d2d286ce20c,23.03,19.14,brinquedos,37.0,512.0,1.0,400.0,20.0,11.0,16.0,68,67


In [40]:
features = [
    "price",
    "freight_value",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "product_category_name_english"
]

X = df[features]
y = df["next_category"]

In [41]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [42]:
joblib.dump(scaler, "scaler3.pkl")


['scaler3.pkl']

In [43]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

In [44]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_dev = torch.tensor(X_dev, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.long)
y_dev = torch.tensor(y_dev.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

In [45]:
class BehaviorDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [46]:
train_dataset = BehaviorDataset(X_train, y_train)
dev_dataset = BehaviorDataset(X_dev, y_dev)
test_dataset = BehaviorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

In [47]:
class BehavioralNet(nn.Module):

    def __init__(self, input_size, num_classes):
        super(BehavioralNet, self).__init__()

        self.network = nn.Sequential(

            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.network(x)

In [48]:
input_size = X_train.shape[1]
num_classes = len(np.unique(y))

model = BehavioralNet(input_size, num_classes)

In [49]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [50]:
epochs = 20

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward() 
        optimizer.step() 

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

Epoch 1, Loss: 3.0353220373719605
Epoch 2, Loss: 2.3927638019834245
Epoch 3, Loss: 2.2245232534932566
Epoch 4, Loss: 2.12152899490608
Epoch 5, Loss: 2.051659400646503
Epoch 6, Loss: 2.0072705915996005
Epoch 7, Loss: 1.9589486115581387
Epoch 8, Loss: 1.9443515676718492
Epoch 9, Loss: 1.9029068842039003
Epoch 10, Loss: 1.8802922381149543
Epoch 11, Loss: 1.8612089641801604
Epoch 12, Loss: 1.835223559494857
Epoch 13, Loss: 1.8158752741394462
Epoch 14, Loss: 1.7909069821074768
Epoch 15, Loss: 1.7805949606738247
Epoch 16, Loss: 1.7621522162641798
Epoch 17, Loss: 1.743774291250732
Epoch 18, Loss: 1.7287874123552343
Epoch 19, Loss: 1.7234995530529336
Epoch 20, Loss: 1.7097967784483354


In [51]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in dev_loader:

        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)

        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

accuracy = correct / total

print("Dev Accuracy:", accuracy)

Dev Accuracy: 0.6211755233494364


In [52]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)

        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

print("Test Accuracy:", correct/total)

Test Accuracy: 0.6388888888888888


In [53]:
import joblib
import torch

torch.save(model.state_dict(), 'behavioral_net.pth')
